# Meta-learning experiment notebook

This notebook demonstrates how to run small-scale experiments for MAML and RL² using the code in `src/`.

Notes:
- The notebook is non-executed in the repo to keep it reproducible.
- See `README.md` and `REPORT.md` for usage instructions and the deliverable structure.
- To run: set up a Python environment from `requirements.txt` then run the cells interactively.


# CA27: Meta-Learning in Reinforcement Learning

This notebook implements and evaluates Model-Agnostic Meta-Learning (MAML) and Recurrent Meta-RL (RL²) algorithms for few-shot adaptation in reinforcement learning tasks.

## 1. Import Required Libraries

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import gym
from typing import List

# Import from src
from src.config import MAMLConfig, RL2Config, TaskConfig
from src.tasks import CartPoleTask, MetaLearningTaskDistribution
from src.maml import MAML
from src.rl2 import RL2Trainer
from src.utils import collect_trajectory

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Load Configuration

In [ ]:
# MAML Configuration
maml_config = MAMLConfig(
    obs_dim=4,  # CartPole observation space
    action_dim=2,  # CartPole action space
    hidden_dim=64,
    inner_lr=0.1,
    meta_lr=0.001,
    inner_steps=1,
    gamma=0.99,
    num_meta_iterations=50,  # Reduced for demo
    meta_batch_size=5,
    num_steps_per_task=50
)

# RL² Configuration
rl2_config = RL2Config(
    obs_dim=4,
    action_dim=2,
    hidden_dim=256,
    num_lstm_layers=2,
    lr=1e-3,
    gamma=0.99,
    lam=0.95,
    num_meta_iterations=30,  # Reduced for demo
    meta_batch_size=3,
    num_episodes_per_task=5,
    ppo_epochs=4,
    clip_ratio=0.2
)

# Task Configuration
task_config = TaskConfig(
    env_name="CartPole-v1",
    num_tasks=20,  # Reduced for demo
    gravity_range=(8.0, 11.0),
    masscart_range=(0.8, 1.2),
    masspole_range=(0.08, 0.12),
    length_range=(0.4, 0.6)
)

print("Configurations loaded:")
print(f"MAML: {maml_config}")
print(f"RL²: {rl2_config}")
print(f"Tasks: {task_config}")

## 3. Define Tasks

In [ ]:
# Create task distribution for meta-learning
task_distribution = MetaLearningTaskDistribution(
    lambda: CartPoleTask(
        gravity=np.random.uniform(*task_config.gravity_range),
        masscart=np.random.uniform(*task_config.masscart_range),
        masspole=np.random.uniform(*task_config.masspole_range),
        length=np.random.uniform(*task_config.length_range)
    ),
    num_tasks=task_config.num_tasks
)

# Sample a few tasks for demonstration
sample_tasks = task_distribution.sample(3)
print(f"Created task distribution with {task_config.num_tasks} tasks")
print("Sample task parameters:")
for i, task in enumerate(sample_tasks):
    print(f"Task {i+1}: gravity={task.gravity:.2f}, masscart={task.masscart:.2f}, masspole={task.masspole:.3f}, length={task.length:.2f}")

# Test tasks for evaluation (different from training tasks)
test_task_distribution = MetaLearningTaskDistribution(
    lambda: CartPoleTask(
        gravity=np.random.uniform(*task_config.gravity_range),
        masscart=np.random.uniform(*task_config.masscart_range),
        masspole=np.random.uniform(*task_config.masspole_range),
        length=np.random.uniform(*task_config.length_range)
    ),
    num_tasks=10
)
test_tasks = test_task_distribution.sample(10)

## 4. Initialize MAML Algorithm

In [ ]:
# Initialize MAML agent
maml_agent = MAML(maml_config)
print("MAML agent initialized")
print(f"Policy network: {maml_agent.policy}")
print(f"Number of parameters: {sum(p.numel() for p in maml_agent.policy.parameters())}")

## 5. Initialize RL² Algorithm

In [ ]:
# Initialize RL² agent
rl2_agent = RL2Trainer(rl2_config)
print("RL² agent initialized")
print(f"Policy network: {rl2_agent.policy}")
print(f"Number of parameters: {sum(p.numel() for p in rl2_agent.policy.parameters())}")

## 6. Train MAML Model

In [ ]:
print("Training MAML...")
maml_losses = maml_agent.train(
    task_distribution,
    num_meta_iterations=maml_config.num_meta_iterations,
    meta_batch_size=maml_config.meta_batch_size
)
print("MAML training completed!")

# Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(maml_losses, label='MAML Meta Loss')
plt.xlabel('Meta Iteration')
plt.ylabel('Loss')
plt.title('MAML Training Loss')
plt.legend()
plt.grid(True)
plt.savefig('pictures/maml_training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Train RL² Model

In [ ]:
print("Training RL²...")
rl2_losses = rl2_agent.train(
    task_distribution,
    num_meta_iterations=rl2_config.num_meta_iterations,
    meta_batch_size=rl2_config.meta_batch_size
)
print("RL² training completed!")

# Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(rl2_losses, label='RL² Meta Loss', color='orange')
plt.xlabel('Meta Iteration')
plt.ylabel('Loss')
plt.title('RL² Training Loss')
plt.legend()
plt.grid(True)
plt.savefig('pictures/rl2_training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Evaluate Models

In [ ]:
print("Evaluating MAML on test tasks...")
maml_rewards = []
for i, task in enumerate(test_tasks):
    reward = maml_agent.adapt_and_evaluate(task, adaptation_steps=1, eval_steps=200)
    maml_rewards.append(reward)
    print(f"Task {i+1}: {reward:.2f}")

print("Evaluating RL² on test tasks...")
rl2_rewards = []
for i, task in enumerate(test_tasks):
    trajectory = rl2_agent.collect_trajectory_rl2(task.env, max_steps=200)
    reward = trajectory['rewards'].sum().item()
    rl2_rewards.append(reward)
    print(f"Task {i+1}: {reward:.2f}")

# Compare results
print("\nEvaluation Results:")
print(f"MAML - Mean: {np.mean(maml_rewards):.2f}, Std: {np.std(maml_rewards):.2f}")
print(f"RL²  - Mean: {np.mean(rl2_rewards):.2f}, Std: {np.std(rl2_rewards):.2f}")

# Save results
results = {
    'maml_rewards': maml_rewards,
    'rl2_rewards': rl2_rewards,
    'maml_mean': np.mean(maml_rewards),
    'maml_std': np.std(maml_rewards),
    'rl2_mean': np.mean(rl2_rewards),
    'rl2_std': np.std(rl2_rewards)
}

np.save('results/evaluation_results.npy', results)

## 9. Visualize Results

In [ ]:
# Plot comparison of algorithms
plt.figure(figsize=(12, 8))

# Training losses
plt.subplot(2, 2, 1)
plt.plot(maml_losses, label='MAML', color='blue')
plt.plot(rl2_losses, label='RL²', color='orange')
plt.xlabel('Meta Iteration')
plt.ylabel('Loss')
plt.title('Training Losses')
plt.legend()
plt.grid(True)

# Test rewards comparison
plt.subplot(2, 2, 2)
algorithms = ['MAML', 'RL²']
means = [np.mean(maml_rewards), np.mean(rl2_rewards)]
stds = [np.std(maml_rewards), np.std(rl2_rewards)]
plt.bar(algorithms, means, yerr=stds, capsize=5, color=['blue', 'orange'], alpha=0.7)
plt.ylabel('Average Reward')
plt.title('Test Performance Comparison')
plt.grid(True, axis='y')

# Individual task rewards
plt.subplot(2, 2, 3)
plt.plot(maml_rewards, 'o-', label='MAML', color='blue', alpha=0.7)
plt.plot(rl2_rewards, 's-', label='RL²', color='orange', alpha=0.7)
plt.xlabel('Test Task')
plt.ylabel('Reward')
plt.title('Per-Task Performance')
plt.legend()
plt.grid(True)

# Reward distribution
plt.subplot(2, 2, 4)
plt.hist(maml_rewards, alpha=0.7, label='MAML', bins=10, color='blue')
plt.hist(rl2_rewards, alpha=0.7, label='RL²', bins=10, color='orange')
plt.xlabel('Reward')
plt.ylabel('Frequency')
plt.title('Reward Distribution')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('pictures/algorithm_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Results saved to:")
print("- pictures/maml_training_loss.png")
print("- pictures/rl2_training_loss.png")
print("- pictures/algorithm_comparison.png")
print("- results/evaluation_results.npy")